# 01. 이미지 수집 · 정제 (YOLOv8 과제)

이 노트북은 Bing 이미지 검색으로 부류별 사진을 모으고, 깨진 파일·너무 작은 사진·중복 사진을 자동으로 걸러낸 뒤,
사람이 눈으로 검수할 수 있게 번호가 붙은 미리보기(contact sheet)를 만들어 줍니다.

**사용 순서**
1. 셀 2에서 `TOPIC`을 확인합니다 (기본값 `"hornets"` = 꿀벌·장수말벌·등검은말벌).
2. 위에서부터 차례로 실행합니다. 수집은 몇 분 걸립니다.
3. 미리보기에서 잘못된 사진(일러스트, 다른 동물, 글자 위주 이미지 등)의 번호를 셀 6의 `DELETE`에 적고 실행합니다.
4. 마지막 셀에서 `dataset_clean.zip`(Roboflow 업로드용)과 `sources.csv`(출처 기록, 보고서용)를 내려받습니다.

`hard_negative` 폴더는 **헷갈리기 쉬운 다른 곤충**(예: 꿀벌을 흉내 내는 꽃등에, 쌍살벌), `background` 폴더는 **동물이 없는 배경**입니다.
Roboflow에서 이 사진들은 박스를 치지 않고 **Null(빈 이미지)** 로 표시합니다. 모델이 이것들을 잘못 탐지하지 않도록 가르치는 용도입니다.

In [ ]:
!pip -q install icrawler imagehash

In [ ]:
# ===== 셀 2: 설정 =====
TOPIC = "hornets"       # "hornets"(꿀벌·장수말벌·등검은말벌) / "invasive"(생태계 교란 생물) / "birds"(도심 새)
MAX_PER_KEYWORD = 120   # 키워드 하나당 최대 다운로드 수

PRESETS = {
    "hornets": {
        "classes": {
            "honeybee": ["honey bee Apis mellifera", "honeybees at hive entrance", "honey bee flying", "양봉꿀벌"],
            "giant_hornet": ["Vespa mandarinia", "Asian giant hornet", "giant hornet attacking beehive", "장수말벌 곤충"],
            "yellow_legged_hornet": ["Vespa velutina nigrithorax", "yellow-legged hornet", "Asian hornet hawking beehive", "등검은말벌"],
        },
        # 꿀벌·말벌과 헷갈리기 쉬운 곤충 → 박스 없이 Null로 업로드
        "hard_negative": ["hoverfly on flower", "paper wasp Polistes", "bumblebee on flower", "housefly macro"],
        "background": ["empty wooden beehive box", "flower meadow close up", "apiary landscape"],
    },
    "invasive": {
        "classes": {
            "bullfrog": ["American bullfrog", "Lithobates catesbeianus bullfrog", "bullfrog in pond", "황소개구리"],
            "red_eared_slider": ["red-eared slider turtle", "Trachemys scripta elegans", "red-eared slider basking on log", "붉은귀거북"],
            "nutria": ["nutria coypu", "Myocastor coypus", "nutria swimming river", "뉴트리아"],
        },
        # 생김새가 비슷한 토종·보호종 → 교란종으로 오탐하면 안 되는 대상
        "hard_negative": ["Eurasian otter", "Reeves turtle Mauremys reevesii", "Chinese softshell turtle",
                          "black-spotted pond frog", "muskrat", "beaver swimming"],
        "background": ["pond water surface", "river bank reeds", "wetland stream rocks"],
    },
    "birds": {
        "classes": {
            "magpie": ["Oriental magpie", "Korean magpie bird", "magpie flying", "까치 새"],
            "pigeon": ["feral pigeon", "rock dove city street", "pigeons flock street", "비둘기 새"],
            "sparrow": ["Eurasian tree sparrow", "tree sparrow perched", "sparrow flying", "참새 새"],
        },
        "hard_negative": ["large-billed crow", "Eurasian jay bird", "white wagtail"],
        "background": ["empty city park", "tree branches sky", "empty sidewalk"],
    },
}

CFG = PRESETS[TOPIC]
RAW, CLEAN, REVIEW = "/content/raw", "/content/clean", "/content/review"
print("TOPIC:", TOPIC, "| 부류:", list(CFG["classes"]))

In [ ]:
# ===== 셀 3: 수집 =====
import os, csv, logging, threading
from icrawler import ImageDownloader
from icrawler.builtin import BingImageCrawler

LOG_ROWS = []            # (raw_path, url) 출처 기록
DIR2KW = {}              # 폴더 → (그룹, 키워드)
_lock = threading.Lock()

class LoggingDownloader(ImageDownloader):
    """다운로드에 성공한 파일의 원본 URL을 기록한다."""
    def process_meta(self, task):
        if task.get("success") and task.get("filename"):
            with _lock:
                LOG_ROWS.append([os.path.join(self.storage.root_dir, task["filename"]), task["file_url"]])

def crawl_group(group, keywords):
    for i, kw in enumerate(keywords):
        out = os.path.join(RAW, group, f"kw{i:02d}")
        DIR2KW[out] = (group, kw)
        crawler = BingImageCrawler(downloader_cls=LoggingDownloader, downloader_threads=4,
                                   storage={"root_dir": out}, log_level=logging.ERROR)
        try:
            crawler.crawl(keyword=kw, max_num=MAX_PER_KEYWORD, filters={"type": "photo"})
        except Exception as e:
            print("   필터 없이 재시도:", e)
            crawler.crawl(keyword=kw, max_num=MAX_PER_KEYWORD)
        n = len(os.listdir(out)) if os.path.isdir(out) else 0
        print(f"  [{group}] '{kw}': {n}장")

for cls, kws in CFG["classes"].items():
    crawl_group(cls, kws)
crawl_group("hard_negative", CFG["hard_negative"])
crawl_group("background", CFG["background"])
print("수집 완료. 기록된 URL:", len(LOG_ROWS))

In [ ]:
# ===== 셀 4: 자동 정제 (깨진 파일 / 작은 사진 / 극단적 비율 / 중복 제거) =====
import glob, shutil
from PIL import Image, ImageOps
import imagehash

MIN_SIDE   = 320    # 짧은 변이 이보다 작으면 제외
MAX_ASPECT = 3.0    # 가로세로 비율이 이보다 크면 제외 (배너·파노라마)
MAX_SIDE   = 1600   # 업로드 용량을 줄이기 위해 긴 변을 이 크기로 축소
HASH_DIST  = 6      # 지각 해시 거리 ≤ 이 값이면 중복으로 판단

EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif")
shutil.rmtree(CLEAN, ignore_errors=True)
seen_hashes, CLEAN_MAP, stats = [], [], []

def is_duplicate(h):
    return any(bin(h ^ s).count("1") <= HASH_DIST for s in seen_hashes)

groups = list(CFG["classes"]) + ["hard_negative", "background"]
for group in groups:
    src, dst = os.path.join(RAW, group), os.path.join(CLEAN, group)
    os.makedirs(dst, exist_ok=True)
    kept = bad = small = dup = 0
    for p in sorted(glob.glob(os.path.join(src, "**", "*"), recursive=True)):
        if not p.lower().endswith(EXTS):
            continue
        try:
            with Image.open(p) as im:
                im = ImageOps.exif_transpose(im).convert("RGB")
        except Exception:
            bad += 1; continue
        w, h = im.size
        if min(w, h) < MIN_SIDE or max(w, h) / min(w, h) > MAX_ASPECT:
            small += 1; continue
        hv = int(str(imagehash.phash(im)), 16)
        if is_duplicate(hv):
            dup += 1; continue
        seen_hashes.append(hv)
        im.thumbnail((MAX_SIDE, MAX_SIDE))
        kept += 1
        out = os.path.join(dst, f"{group}_{kept:04d}.jpg")
        im.save(out, "JPEG", quality=92)
        CLEAN_MAP.append([out, p])
    stats.append((group, kept, bad, small, dup))

print(f"{'그룹':<18}{'남음':>6}{'깨짐':>6}{'작음/비율':>10}{'중복':>6}")
for g, k, b, s, d in stats:
    print(f"{g:<18}{k:>6}{b:>6}{s:>10}{d:>6}")

In [ ]:
# ===== 셀 5: 검수용 미리보기 (번호 확인 후 셀 6에 적기) =====
from PIL import ImageDraw
from IPython.display import display

def contact_sheets(group, cols=8, rows=6, thumb=170):
    os.makedirs(REVIEW, exist_ok=True)
    files = sorted(glob.glob(os.path.join(CLEAN, group, "*.jpg")))
    per = cols * rows
    for start in range(0, len(files), per):
        sheet = Image.new("RGB", (cols * thumb, rows * (thumb + 20)), "white")
        draw = ImageDraw.Draw(sheet)
        for k, f in enumerate(files[start:start + per]):
            with Image.open(f) as im:
                im.thumbnail((thumb, thumb))
                x, y = (k % cols) * thumb, (k // cols) * (thumb + 20)
                sheet.paste(im, (x + (thumb - im.width) // 2, y))
            num = int(os.path.basename(f).rsplit("_", 1)[1].split(".")[0])
            draw.text((x + 4, y + thumb + 3), str(num), fill="black")
        path = os.path.join(REVIEW, f"{group}_p{start // per + 1}.jpg")
        sheet.save(path, quality=85)
        print(path); display(sheet)

for group in groups:
    print(f"\n===== {group} =====")
    contact_sheets(group)

In [ ]:
# ===== 셀 6: 잘못된 사진 삭제 =====
# 미리보기 번호를 적고 실행하세요. 예) "giant_hornet": [3, 17, 42]
DELETE = {
    # "honeybee": [],
    # "giant_hornet": [],
    # "yellow_legged_hornet": [],
    # "hard_negative": [],
    # "background": [],
}

removed = 0
for group, nums in DELETE.items():
    for n in nums:
        f = os.path.join(CLEAN, group, f"{group}_{n:04d}.jpg")
        if os.path.exists(f):
            os.remove(f); removed += 1
print("삭제:", removed, "장")
for group in groups:
    print(f"  {group:<18}{len(glob.glob(os.path.join(CLEAN, group, '*.jpg'))):>5}장")

In [ ]:
# ===== 셀 7: 출처 기록 저장 + 내려받기 =====
url_of = {raw: url for raw, url in LOG_ROWS}
kw_of = lambda raw: DIR2KW.get(os.path.dirname(raw), ("", ""))
with open("/content/sources.csv", "w", newline="", encoding="utf-8-sig") as fp:
    wr = csv.writer(fp)
    wr.writerow(["clean_file", "group", "keyword", "source_url"])
    for out, raw in CLEAN_MAP:
        if os.path.exists(out):              # 검수에서 지운 파일은 제외
            g, kw = kw_of(raw)
            wr.writerow([os.path.relpath(out, CLEAN), g, kw, url_of.get(raw, "")])

shutil.make_archive("/content/dataset_clean", "zip", CLEAN)
print("dataset_clean.zip, sources.csv 생성 완료")
try:
    from google.colab import files
    files.download("/content/dataset_clean.zip")
    files.download("/content/sources.csv")
except ImportError:
    pass